<a href="https://colab.research.google.com/github/franciscomartinez45/Stock-Forecasting-LSTM/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/ProsusAI/finbert

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ProsusAI/finbert)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [6]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
POLYGON_API_KEY = userdata.get('POLYGON_API_KEY')

In [9]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=HF_TOKEN,
)

result = client.text_classification(
    "I like you. I love you",
    model="ProsusAI/finbert",
)

In [10]:
!pip install -q transformers torch

In [11]:
!wget -q "https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip" -O financial_phrasebank.zip
!unzip -q financial_phrasebank.zip
!ls FinancialPhraseBank-v1.0/

replace FinancialPhraseBank-v1.0/License.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/FinancialPhraseBank-v1.0/._License.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace FinancialPhraseBank-v1.0/README.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace __MACOSX/FinancialPhraseBank-v1.0/._README.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace FinancialPhraseBank-v1.0/Sentences_50Agree.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace FinancialPhraseBank-v1.0/Sentences_66Agree.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace FinancialPhraseBank-v1.0/Sentences_66Agree.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
License.txt  Sentences_50Agree.txt  Sentences_75Agree.txt
README.txt   Sentences_66Agree.txt  Sentences_AllAgree.txt


In [12]:
from transformers import pipeline
import pandas as pd

label_map_str = {"negative": 0, "neutral": 1, "positive": 2}
label_map_int = {0: "negative", 1: "neutral", 2: "positive"}
sentences, true_labels = [], []

with open("FinancialPhraseBank-v1.0/Sentences_AllAgree.txt", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if "@" in line:
            sentence, label = line.rsplit("@", 1)
            sentences.append(sentence.strip())
            true_labels.append(label_map_str[label.strip()])

print(f"Loaded {len(sentences)} samples")

pipe = pipeline("text-classification", model="ProsusAI/finbert", device=0, top_k=None)
results = pipe(sentences[:200], batch_size=16, truncation=True)

rows = []
for sentence, true_label, scores in zip(sentences[:200], true_labels[:200], results):
    score_dict = {s["label"].lower(): round(s["score"], 4) for s in scores}
    rows.append({
        "sentence": sentence,
        "true_label": label_map_int[true_label],
        "negative": score_dict.get("negative", 0),
        "neutral":  score_dict.get("neutral", 0),
        "positive": score_dict.get("positive", 0),
        "predicted_label": max(score_dict, key=score_dict.get),
    })

df = pd.DataFrame(rows)
accuracy = (df["true_label"] == df["predicted_label"]).mean()
print(f"Accuracy on {len(df)} samples: {accuracy:.2%}\n")
print(df[["sentence", "true_label", "negative", "neutral", "positive", "predicted_label"]].head(10).to_string(index=False))

Loaded 2264 samples


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy on 200 samples: 99.50%

                                                                                                                                                                                                                                                      sentence true_label  negative  neutral  positive predicted_label
                                                                                                                               According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .    neutral    0.0109   0.8857    0.1034         neutral
                                                             For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m .   positive    0.0260   0.0258    0.9482        positive
                                  

In [13]:
!pip install -q yfinance

import yfinance as yf

ticker = "AAPL"
df_stock = yf.download(ticker, start="2023-01-01", end="2024-01-01", auto_adjust=True)

print(f"Shape: {df_stock.shape}")
print(f"\nColumns: {list(df_stock.columns)}")
print(f"\nDate range: {df_stock.index[0].date()} → {df_stock.index[-1].date()}")
print("\n", df_stock.head(10))

[*********************100%***********************]  1 of 1 completed

Shape: (250, 5)

Columns: [('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]

Date range: 2023-01-03 → 2023-12-29

 Price            Close        High         Low        Open     Volume
Ticker            AAPL        AAPL        AAPL        AAPL       AAPL
Date                                                                 
2023-01-03  122.982712  128.715409  122.097730  128.105761  112117500
2023-01-04  124.251183  126.512801  122.992546  124.772336   89113600
2023-01-05  122.933540  125.637646  122.677885  125.008327   80962700
2023-01-06  127.456787  128.115604  122.805730  123.907041   87754700
2023-01-09  127.977928  131.183532  127.722273  128.292595   70790800
2023-01-10  128.548264  129.069417  125.981821  128.086106   63896200
2023-01-11  131.262207  131.281862  128.282776  129.059585   69458900
2023-01-12  131.183517  132.019323  129.246394  131.645675   71379600
2023-01-13  132.511002  132.668335  129.462746  129.826566   57809700
20

In [14]:
import requests
import pandas as pd
import time

ticker     = "AAPL"
start_date = df_stock.index[0].strftime("%Y-%m-%d")
end_date   = df_stock.index[-1].strftime("%Y-%m-%d")

url      = "https://api.polygon.io/v2/reference/news"
articles = []
next_url = None
page     = 0

params = {
    "ticker": ticker,
    "published_utc.gte": start_date,
    "published_utc.lte": end_date,
    "order": "asc",
    "limit": 1000,
    "apiKey": POLYGON_API_KEY,
}

while True:
    resp = requests.get(
        next_url or url,
        params=params if not next_url else {"apiKey": POLYGON_API_KEY}
    )
    data = resp.json()
    batch = data.get("results", [])
    articles.extend(batch)
    page += 1
    print(f"Page {page}: fetched {len(batch)} articles (total so far: {len(articles)})")

    next_url = data.get("next_url")
    if not next_url:
        break
    time.sleep(12)  # stay under free-tier limit of 5 req/min

df_news = pd.DataFrame([{
    "date":      a["published_utc"][:10],
    "title":     a["title"],
    "publisher": a["publisher"]["name"],
    "url":       a["article_url"],
} for a in articles])

df_news.to_csv("aapl_news_2023.csv", index=False)

print(f"\nTotal articles: {len(df_news)}")
print(f"Date range: {df_news['date'].min()} → {df_news['date'].max()}")
print(f"Articles per day (mean): {df_news.groupby('date').size().mean():.1f}")
print("Saved to aapl_news_2023.csv")
print()
print(df_news.head(10).to_string(index=False))

Page 1: fetched 1000 articles (total so far: 1000)
Page 2: fetched 1000 articles (total so far: 2000)
Page 3: fetched 1000 articles (total so far: 3000)
Page 4: fetched 1000 articles (total so far: 4000)
Page 5: fetched 1000 articles (total so far: 5000)
Page 6: fetched 133 articles (total so far: 5133)

Total articles: 5133
Date range: 2023-01-03 → 2023-12-29
Articles per day (mean): 14.3
Saved to aapl_news_2023.csv

      date                                                                                                                title                 publisher                                                                                                                                                      url
2023-01-03                                                     Rotate To Value In 2023: 1 Strong Dividend Buy And 1 Urgent Sell             Seeking Alpha                                                        https://seekingalpha.com/article/4567429-rotate-to-value-2023-

In [15]:
import yfinance as yf

df_msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", auto_adjust=True)

print(f"Shape: {df_msft.shape}")
print(f"Date range: {df_msft.index[0].date()} → {df_msft.index[-1].date()}")
print(df_msft.head(10))

[*********************100%***********************]  1 of 1 completed

Shape: (250, 5)
Date range: 2023-01-03 → 2023-12-29
Price            Close        High         Low        Open    Volume
Ticker            MSFT        MSFT        MSFT        MSFT      MSFT
Date                                                                
2023-01-03  233.452805  239.465007  231.328550  236.863293  25740000
2023-01-04  223.240814  226.914386  220.181120  226.339479  50623400
2023-01-05  216.624466  221.730459  216.088529  221.389405  39585600
2023-01-06  219.177444  219.986219  213.740165  217.296811  43613600
2023-01-09  221.311447  225.326089  220.619614  220.658584  27369800
2023-01-10  222.997208  225.394285  221.516077  221.935073  27033900
2023-01-11  229.740265  229.915654  225.199439  225.374829  28669300
2023-01-12  232.410156  233.764607  227.586754  229.243274  27269500
2023-01-13  233.111755  233.248174  228.911985  230.938791  21333300
2023-01-17  234.203125  234.748801  231.026489  231.883988  29831300


In [16]:
import requests
import pandas as pd
import time

msft_start = df_msft.index[0].strftime("%Y-%m-%d")
msft_end   = df_msft.index[-1].strftime("%Y-%m-%d")

url      = "https://api.polygon.io/v2/reference/news"
articles = []
next_url = None
page     = 0

params = {
    "ticker": "MSFT",
    "published_utc.gte": msft_start,
    "published_utc.lte": msft_end,
    "order": "asc",
    "limit": 1000,
    "apiKey": POLYGON_API_KEY,
}

while True:
    resp = requests.get(
        next_url or url,
        params=params if not next_url else {"apiKey": POLYGON_API_KEY}
    )
    data = resp.json()
    batch = data.get("results", [])
    articles.extend(batch)
    page += 1
    print(f"Page {page}: fetched {len(batch)} articles (total so far: {len(articles)})")

    next_url = data.get("next_url")
    if not next_url:
        break
    time.sleep(12)

df_msft_news = pd.DataFrame([{
    "date":      a["published_utc"][:10],
    "title":     a["title"],
    "publisher": a["publisher"]["name"],
    "url":       a["article_url"],
} for a in articles])

df_msft_news.to_csv("msft_news_2023.csv", index=False)

print(f"\nTotal articles: {len(df_msft_news)}")
print(f"Date range: {df_msft_news['date'].min()} → {df_msft_news['date'].max()}")
print(f"Articles per day (mean): {df_msft_news.groupby('date').size().mean():.1f}")
print("Saved to msft_news_2023.csv")
print()
print(df_msft_news.head(10).to_string(index=False))

Page 1: fetched 0 articles (total so far: 0)

Total articles: 0


KeyError: 'date'

In [ ]:
from transformers import pipeline
import pandas as pd

pipe = pipeline("text-classification", model="ProsusAI/finbert", device=0, top_k=None)

trading_dates = df_msft.index.strftime("%Y-%m-%d").tolist()
rows = []

for i, date in enumerate(trading_dates[:-1]):
    next_date = trading_dates[i + 1]
    day_titles = df_msft_news[df_msft_news["date"] == date]["title"].tolist()

    if not day_titles:
        print(f"{date}: no articles, skipping")
        continue

    results = pipe(day_titles, batch_size=16, truncation=True)

    neg, neu, pos = [], [], []
    for scores in results:
        score_dict = {s["label"].lower(): s["score"] for s in scores}
        neg.append(score_dict.get("negative", 0))
        neu.append(score_dict.get("neutral",  0))
        pos.append(score_dict.get("positive", 0))

    close_today = df_msft.loc[date,      "Close"].item()
    close_next  = df_msft.loc[next_date, "Close"].item()
    change_pct  = (close_next - close_today) / close_today * 100

    rows.append({
        "date":          date,
        "n_articles":    len(day_titles),
        "avg_negative":  round(sum(neg) / len(neg), 4),
        "avg_neutral":   round(sum(neu) / len(neu), 4),
        "avg_positive":  round(sum(pos) / len(pos), 4),
        "close":         round(close_today, 2),
        "next_close":    round(close_next,  2),
        "next_day_pct":  round(change_pct,  4),
        "direction":     "UP" if close_next > close_today else "DOWN",
    })
    print(f"{date}: {len(day_titles)} articles | neg={rows[-1]['avg_negative']:.3f} neu={rows[-1]['avg_neutral']:.3f} pos={rows[-1]['avg_positive']:.3f} | {rows[-1]['direction']} ({change_pct:+.2f}%)")

df_msft_daily = pd.DataFrame(rows)
df_msft_daily.to_csv("msft_daily_sentiment_2023.csv", index=False)

print(f"\nDone. {len(df_msft_daily)} trading days processed.")
print("Saved to msft_daily_sentiment_2023.csv")
print()
print(df_msft_daily.head(10).to_string(index=False))

In [ ]:
from transformers import pipeline
import pandas as pd

pipe = pipeline("text-classification", model="ProsusAI/finbert", device=0, top_k=None)

trading_dates = df_stock.index.strftime("%Y-%m-%d").tolist()
rows = []

for i, date in enumerate(trading_dates[:-1]):  # exclude last day (no next-day return)
    next_date = trading_dates[i + 1]
    day_titles = df_news[df_news["date"] == date]["title"].tolist()

    if not day_titles:
        print(f"{date}: no articles, skipping")
        continue

    results = pipe(day_titles, batch_size=16, truncation=True)

    neg, neu, pos = [], [], []
    for scores in results:
        score_dict = {s["label"].lower(): s["score"] for s in scores}
        neg.append(score_dict.get("negative", 0))
        neu.append(score_dict.get("neutral",  0))
        pos.append(score_dict.get("positive", 0))

    close_today = df_stock.loc[date,      "Close"].item()
    close_next  = df_stock.loc[next_date, "Close"].item()
    change_pct  = (close_next - close_today) / close_today * 100

    rows.append({
        "date":           date,
        "n_articles":     len(day_titles),
        "avg_negative":   round(sum(neg) / len(neg), 4),
        "avg_neutral":    round(sum(neu) / len(neu), 4),
        "avg_positive":   round(sum(pos) / len(pos), 4),
        "close":          round(close_today, 2),
        "next_close":     round(close_next,  2),
        "next_day_pct":   round(change_pct,  4),
        "direction":      "UP" if close_next > close_today else "DOWN",
    })
    print(f"{date}: {len(day_titles)} articles | neg={rows[-1]['avg_negative']:.3f} neu={rows[-1]['avg_neutral']:.3f} pos={rows[-1]['avg_positive']:.3f} | {rows[-1]['direction']} ({change_pct:+.2f}%)")

df_daily = pd.DataFrame(rows)
df_daily.to_csv("aapl_daily_sentiment_2023.csv", index=False)

print(f"\nDone. {len(df_daily)} trading days processed.")
print(f"Saved to aapl_daily_sentiment_2023.csv")
print()
print(df_daily.head(10).to_string(index=False))